In [4]:
import pandas as pd
import time
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder

# --- APRIORI PENTRU ROMÂNIA ---

try:
    print("--- Pregătire date Apriori (România) ---")
    # 1. Încărcăm datele curate (fără apă)
    df_ro = pd.read_csv('Romania_foodex_clean.csv')
    
    # 2. Transformare în formatul necesar: Listă de Liste
    # Apriori are nevoie de o structură: [['pâine', 'unt'], ['pâine', 'lapte'], ...]
    transactions = df_ro.groupby('Transaction_ID')['Item'].apply(list).tolist()
    
    print(f"Număr tranzacții: {len(transactions)}")
    
    # 3. One-Hot Encoding (Matricea True/False)
    # Acesta este pasul "greu" pentru memorie
    prep_start = time.time()
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
    
    prep_time = time.time() - prep_start
    print(f"Codificare One-Hot finalizată în {prep_time:.2f} secunde.")
    print(f"Dimensiune Matrice: {df_encoded.shape}")

    # 4. Rulare Apriori
    print("Se rulează Apriori...")
    apriori_start = time.time()
    
    # Folosim același min_support ca la ECLAT (0.05 sau 0.02, verifică ce ai folosit anterior)
    frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)
    
    apriori_time = time.time() - apriori_start
    print(f"Gata! Apriori a terminat în {apriori_time:.4f} secunde.")
    
    # 5. Afișare Rezultate
    # Adăugăm o coloană cu lungimea setului pentru filtrare
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
    
    # Filtrăm doar combinațiile (lungime >= 2)
    top_rules = frequent_itemsets[frequent_itemsets['length'] >= 2].sort_values(by='support', ascending=False).head(15)
    
    print("\n--- Top Asocieri Apriori (România) ---")
    if top_rules.empty:
        print("Nicio combinație găsită. Încearcă un support mai mic (ex: 0.02).")
    else:
        print(top_rules[['itemsets', 'support']])

except FileNotFoundError:
    print("Eroare: Nu găsesc 'Romania_foodex_clean.csv'.")
except MemoryError:
    print("Eroare Memorie: Setul de date e prea mare pentru Apriori. Încearcă să crești min_support.")

--- Pregătire date Apriori (România) ---
Număr tranzacții: 9664
Codificare One-Hot finalizată în 0.04 secunde.
Dimensiune Matrice: (9664, 397)
Se rulează Apriori...
Gata! Apriori a terminat în 89.2127 secunde.

--- Top Asocieri Apriori (România) ---
                                               itemsets   support
853                          (sunflower seed oil, salt)  0.858651
734                                      (onions, salt)  0.829367
864                       (salt, wheat bread and rolls)  0.798634
310                                     (carrots, salt)  0.773903
736                        (onions, sunflower seed oil)  0.767384
4852                 (onions, sunflower seed oil, salt)  0.762210
298                                   (onions, carrots)  0.758071
2569                            (onions, carrots, salt)  0.753725
882         (sunflower seed oil, wheat bread and rolls)  0.736238
5281  (sunflower seed oil, salt, wheat bread and rolls)  0.727752
747                     

In [5]:
# --- APRIORI PENTRU NIGERIA ---

try:
    print("--- Pregătire date Apriori (Nigeria) ---")
    # 1. Încărcăm datele
    df_ng = pd.read_csv('Nigeria_foodex_clean.csv')
    
    # 2. Transformare Listă de Liste
    transactions = df_ng.groupby('Transaction_ID')['Item'].apply(list).tolist()
    
    print(f"Număr tranzacții: {len(transactions)}")
    
    # 3. One-Hot Encoding
    prep_start = time.time()
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
    
    print(f"Codificare gata în {time.time() - prep_start:.2f} secunde.")
    print(f"Dimensiune Matrice: {df_encoded.shape}")

    # 4. Rulare Apriori
    print("Se rulează Apriori...")
    apriori_start = time.time()
    
    # Păstrăm consistența suportului (0.05)
    frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)
    
    apriori_time = time.time() - apriori_start
    print(f"Gata! Apriori a terminat în {apriori_time:.4f} secunde.")
    
    # 5. Afișare
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
    top_rules = frequent_itemsets[frequent_itemsets['length'] >= 2].sort_values(by='support', ascending=False).head(15)
    
    print("\n--- Top Asocieri Apriori (Nigeria) ---")
    if top_rules.empty:
        print("Nicio combinație găsită. Încearcă un support mai mic.")
    else:
        print(top_rules[['itemsets', 'support']])

except FileNotFoundError:
    print("Eroare: Nu găsesc 'Nigeria_foodex_clean.csv'.")

--- Pregătire date Apriori (Nigeria) ---
Număr tranzacții: 990
Codificare gata în 0.00 secunde.
Dimensiune Matrice: (990, 141)
Se rulează Apriori...
Gata! Apriori a terminat în 0.0120 secunde.

--- Top Asocieri Apriori (Nigeria) ---
                                             itemsets   support
53               (mammals and birds meat, rice grain)  0.303030
66                 (rice grain, soups (ready-to-eat))  0.294949
57     (mammals and birds meat, soups (ready-to-eat))  0.290909
37              (cassava roots, soups (ready-to-eat))  0.210101
76      (soups (ready-to-eat), wheat bread and rolls)  0.166667
98  (mammals and birds meat, rice grain, soups (re...  0.156566
50                          (rice grain, fish (meat))  0.156566
47  (dried starchy roots and tuber products, soups...  0.155556
78                       (yams, soups (ready-to-eat))  0.154545
58    (mammals and birds meat, wheat bread and rolls)  0.153535
52                (fish (meat), soups (ready-to-eat))  0.146465